# The axis of evil in the microwave sky

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/axis_of_evil)

The two largest patterns on the real WMAP map of the cosmic microwave background, the quadrupole ($\ell = 2$) and octopole ($\ell = 3$), seem to point the same way. Their coefficients $a_{\ell m}$ come from one least-squares fit; each multipole's axis is the top eigenvector of the $3\times 3$ angular-momentum dispersion $M_{ij} = \mathrm{Re}\,a^*L_iL_ja$, and a batch of $10^5$ isotropic random skies turns the angle between the axes into a $p$-value. A real but debated anomaly: chance, foreground residue or new physics.

**Vector form**

$$T(\hat n) = \sum_{\ell m} a_{\ell m}\,Y_{\ell m}(\hat n),\qquad \hat n_\ell = \arg\max_{\hat n}\ \langle a_\ell|(\hat n\cdot\mathbf L)^2|a_\ell\rangle,\qquad S = |\hat n_2\cdot\hat n_3|,\qquad p = P_{\mathrm{iso}}(S \ge S_{\mathrm{obs}})$$

**Index (tensor) form**

$$r = \arg\min\ w_p\big(T_p - Y_{pa}\,r_a\big)^2,\qquad M^{(\ell)}_{ij} = \mathrm{Re}\;a^*_{\ell m}\,(L_i)_{mn}(L_j)_{nk}\,a_{\ell k},\qquad M_{ij}\hat n_j = \lambda_{\max}\hat n_i,\qquad S_s = |\hat n_{2,si}\,\hat n_{3,si}|$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\hat n_p,\ T_p,\ w_p$ | `load_sky_map(path)` | (P, 3), (P,), (P,) | pixel direction, temperature and solid angle, from any lon/lat/T CSV |
| $Y_{pa}$ | `real_harmonics(dirs, lmax)` | (P, (\ell_\mathrm{fit}+1)^2) | real harmonics at every pixel; $a$ runs over all $(\ell, m)$ |
| $r_a$ | `lstsq(Y * sqrt(w), T * sqrt(w))` | ((\ell_\mathrm{fit}+1)^2,) | weighted least squares: all multipoles at once, no aliasing between them |
| $a_{\ell m}$ | `real_to_complex(r, ell)` | (2\ell+1,) | complex coefficients with $a_{\ell,-m} = (-1)^m a^*_{\ell m}$ |
| $(L_i)_{mn}$ | `angular_momentum(ell)` | (3, 2\ell+1, 2\ell+1) | spin-$\ell$ matrices, $[L_x, L_y] = iL_z$, $L_iL_i = \ell(\ell+1)$ |
| $M_{ij}$ | `einsum('...m,imn,jnk,...k->...ij', a.conj(), L, L, a).real` | (K, 3, 3) | $m, n, k$ summed: $\hat n_iM_{ij}\hat n_j = \sum_m m^2/a_{\ell m}/^2$ in the frame with $z \parallel \hat n$ |
| $\hat n_\ell$ | `eigh(M).eigenvectors[..., -1]` | (K, 3) | batched eigh: the axis around which the multipole is most planar |
| $S_s$ | `einsum('si,si->s', n2, n3).abs()` | (K,) | $i$ summed: alignment of the two axes in sky $s$ (axes are headless) |
| $p$ | `(S_sims >= S_obs).mean()` | () | Monte Carlo $p$-value against isotropic Gaussian skies |

In a statistically isotropic universe (the standard inflationary prediction) the orientations of different multipoles are independent, so the angle between the quadrupole and octopole axes should be random: $|\cos|$ uniform on $[0, 1]$. Tegmark, de Oliveira-Costa and Hamilton (2003) and de Oliveira-Costa et al. (2004) found in the first WMAP maps that both point to $(l, b) \approx (-110^\circ, 60^\circ)$, nicknamed the "axis of evil" by Land and Magueijo (2005); Copi, Huterer, Schwarz and Starkman studied it with multipole vectors and found further links to the ecliptic. On the bundled WMAP 9-year ILC map the axes come out at $(l, b) \approx (240^\circ, 60^\circ)$ and $(238^\circ, 63^\circ)$, only $3^\circ$ apart, and fewer than 2 in 1000 isotropic skies do better ($p \approx 0.0015$; for this statistic $p = 1 - |\cos|$ exactly, which the Monte Carlo confirms). The right panel checks the simulation: the share of random skies at least as aligned as $x$ follows the line $P = x$. Status: the alignment is real in the data (Planck 2013, 2015 and 2018 isotropy papers see it in all their maps), but its significance is debated. The statistic was chosen after the pattern was noticed (a posteriori, look-elsewhere effect: among many possible anomalies some will look rare); the low quadrupole power and the ILC's residual Galactic foregrounds, the kinematic Doppler quadrupole (not removed here) and the choice of map shift $p$ by factors of several. No accepted physical explanation exists; proposals include a non-trivial topology, anisotropic expansion or local foregrounds, and most cosmologists treat it as a statistical fluke. This page reports $p$ for one statistic and does not claim a detection. Data: WMAP 9-year ILC map (Bennett et al. 2013, ApJS 208, 20), file wmap_ilc_9yr_v5.fits from NASA's LAMBDA archive (lambda.gsfc.nasa.gov; NASA data, public domain), degraded here from HEALPix $N_\mathrm{side} = 512$ to 16 (columns glon_deg, glat_deg, T_uK). Any map in the same lon/lat/T CSV form (for example a Planck map) can replace it. The fit uses all multipoles up to $\ell_\mathrm{fit}$ on the full sky without a mask; isotropic skies use unit Gaussian real coefficients (the power $C_\ell$ cancels in the axis).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/axis_of_evil/main.py
"""Main loop: sky map → a_ℓm by least squares → quadrupole and octopole axes → Monte Carlo significance."""

from pathlib import Path

import torch

from problems._common.runtime import cli, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # CPU, float64: the map is tiny (3072 pixels) and eigh of 3×3 matrices is cheap even for 10⁵ skies.
    fast = fast_mode()
    gen = torch.Generator().manual_seed(params["seed"])
    dirs, T, w = tensors.load_sky_map()
    print(f"{tensors.DATA.name}: {len(T)} pixels, rms {T.std():.1f} μK")

    alm = solve.fit_alm(dirs, T, w, params["lmax_fit"])
    D = solve.power(alm)
    print(
        f"band powers ℓ(ℓ+1)C_ℓ/2π: ℓ=2 {D[2]:.0f} μK², ℓ=3 {D[3]:.0f} μK²  (the Planck ΛCDM fit expects ≈ 1000 each)"
    )
    res = solve.alignment_of(alm)
    for ell, key in ((2, "axis2"), (3, "axis3")):
        lon, lat = tensors.lonlat(res[key])
        print(f"ℓ = {ell} axis: (l, b) = ({lon:.1f}°, {lat:+.1f}°)   planarity t = {res[f't{ell}']:.3f}")
    print(f"angle between the axes: {res['angle']:.2f}°   |n̂2·n̂3| = {res['cos']:.5f}")

    n_sims = 5000 if fast else params["n_skies"]
    cos_sims = solve.monte_carlo(n_sims, gen)
    p = solve.p_value(res["cos"], cos_sims)
    print(f"{n_sims} isotropic Gaussian skies: mean |cos| = {cos_sims.mean():.4f} (uniform: 0.5)")
    print(f"p-value P(|cos| ≥ observed) = {p:.5f}   (exact for isotropy: 1 − |cos| = {1 - res['cos']:.5f})")
    t3_sims = solve.axis_of(tensors.isotropic_coefficients(n_sims, 3, gen), 3)[1]
    print(f"octopole planarity t = {res['t3']:.3f}: exceeded by {solve.p_value(res['t3'], t3_sims):.4f} of the skies")
    print("Caveat: the statistic was chosen after seeing the data (look-elsewhere), so the p-value overstates it.")
    return plot.animate(alm, res, cos_sims, out_dir / "axis_of_evil.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/axis_of_evil/equation.py
r"""Physics → tensor translation: spherical harmonics, angular momentum and the preferred axis of a multipole.

The temperature of the microwave sky is expanded in spherical harmonics,

    T(\hat n) = \sum_{ℓm} a_{ℓm}\,Y_{ℓm}(\hat n),        a_{ℓ,−m} = (−1)^m a^*_{ℓm}  (T is real).

Each multipole ℓ is a vector a_m with 2ℓ+1 complex components, on which the angular-momentum matrices L_i of spin ℓ
act. De Oliveira-Costa et al. (2004) define the axis of a multipole as the direction \hat n around which the
power is most "planar" (concentrated in m = ±ℓ), i.e. the one that maximises the angular-momentum dispersion

    \sum_m m^2 |a_{ℓm}(\hat n)|^2 = \langle a | (\hat n\cdot L)^2 | a\rangle = \hat n_i\,M_{ij}\,\hat n_j,
    M_{ij} = \mathrm{Re}\; a^*_m\,(L_i L_j)_{mm'}\,a_{m'}       (m, m' summed).

So the axis is simply the top eigenvector of a 3 × 3 symmetric matrix, and a batch of skies is a batch of eigh.
The alignment statistic is |\hat n_2 \cdot \hat n_3| (axes have no head or tail).
"""

import math

import numpy as np
import torch


def gauss_legendre(n: int):
    """Nodes and weights of n-point Gauss–Legendre quadrature on [−1, 1] (for exact sphere integrals)."""
    x, w = np.polynomial.legendre.leggauss(n)
    return torch.tensor(x), torch.tensor(w)


def index(ell: int, m: int) -> int:
    """Position of (ℓ, m) in the flat coefficient vector: ℓ = 0 first, m = −ℓ … ℓ within each ℓ."""
    return ell * ell + ell + m


def spherical_harmonics(n: torch.Tensor, lmax: int) -> torch.Tensor:
    r"""Complex Y_{ℓm}(\hat n) with the Condon–Shortley phase.   n: (P, 3) unit vectors → (P, (lmax+1)²)

    The associated Legendre functions P_ℓ^m(cos θ) come from the standard three-term recurrence in ℓ; the loops
    run over (ℓ, m) only, every pixel is handled at once.
    """
    x = n[:, 2].clamp(-1, 1)  # cos θ
    s = torch.sqrt((1 - x * x).clamp_min(0))  # sin θ
    phi = torch.atan2(n[:, 1], n[:, 0])
    Y = torch.zeros(n.shape[0], (lmax + 1) ** 2, dtype=torch.complex128, device=n.device)
    for m in range(lmax + 1):
        p_mm = (-1) ** m * float(math.prod(range(1, 2 * m, 2))) * s**m  # P_m^m = (−1)^m (2m−1)!! sin^m θ
        p_prev, p = torch.zeros_like(x), p_mm
        for ell in range(m, lmax + 1):
            if ell > m:  # P_ℓ^m from P_{ℓ−1}^m and P_{ℓ−2}^m
                p_prev, p = p, ((2 * ell - 1) * x * p - (ell + m - 1) * p_prev) / (ell - m)
            norm = math.sqrt((2 * ell + 1) / (4 * math.pi) * math.factorial(ell - m) / math.factorial(ell + m))
            y = norm * p * torch.exp(1j * m * phi)
            Y[:, index(ell, m)] = y
            Y[:, index(ell, -m)] = (-1) ** m * y.conj()  # Y_{ℓ,−m} = (−1)^m Y*_{ℓm}
    return Y


def real_harmonics(n: torch.Tensor, lmax: int) -> torch.Tensor:
    r"""Real orthonormal harmonics: Y_{ℓ0}, and \sqrt2(−1)^m Re / Im Y_{ℓm} for m > 0 / m < 0.   → (P, (lmax+1)²)"""
    Y = spherical_harmonics(n, lmax)
    R = torch.empty(Y.shape, dtype=torch.float64, device=n.device)
    for ell in range(lmax + 1):
        R[:, index(ell, 0)] = Y[:, index(ell, 0)].real
        for m in range(1, ell + 1):
            R[:, index(ell, m)] = math.sqrt(2) * (-1) ** m * Y[:, index(ell, m)].real
            R[:, index(ell, -m)] = math.sqrt(2) * (-1) ** m * Y[:, index(ell, m)].imag
    return R


def real_to_complex(r: torch.Tensor, ell: int) -> torch.Tensor:
    r"""a_{ℓm} from real coefficients r_{ℓm} (…, 2ℓ+1): a_{ℓm} = (−1)^m (r_{ℓm} − i r_{ℓ,−m})/\sqrt2 for m > 0."""
    a = torch.zeros(r.shape, dtype=torch.complex128, device=r.device)
    a[..., ell] = r[..., ell]
    for m in range(1, ell + 1):
        a_m = (-1) ** m * (r[..., ell + m] - 1j * r[..., ell - m]) / math.sqrt(2)
        a[..., ell + m] = a_m
        a[..., ell - m] = (-1) ** m * a_m.conj()
    return a


def angular_momentum(ell: int) -> torch.Tensor:
    r"""Spin-ℓ matrices (L_x, L_y, L_z) in the basis m = −ℓ … ℓ: shape (3, 2ℓ+1, 2ℓ+1), complex.

    L_z = diag(m),  L_+|m⟩ = \sqrt{ℓ(ℓ+1) − m(m+1)}\,|m+1⟩,  L_x = (L_+ + L_−)/2,  L_y = (L_+ − L_−)/2i.
    """
    m = torch.arange(-ell, ell + 1, dtype=torch.float64)
    up = torch.sqrt(ell * (ell + 1) - m[:-1] * (m[:-1] + 1))  # ⟨m+1|L_+|m⟩
    Lp = torch.diag(up, -1).to(torch.complex128)  # row m+1, column m
    Lm = Lp.T.conj()
    return torch.stack([(Lp + Lm) / 2, (Lp - Lm) / 2j, torch.diag(m).to(torch.complex128)])


def dispersion_tensor(a: torch.Tensor, ell: int) -> torch.Tensor:
    r"""M_{ij} = Re a^*_m (L_i L_j)_{mm'} a_{m'} / Σ|a|², symmetrised.   a: (…, 2ℓ+1) → (…, 3, 3)"""
    L = angular_momentum(ell).to(a.device)
    M = torch.einsum("...m,imn,jnk,...k->...ij", a.conj(), L, L, a).real  # m, n, k summed
    M = (M + M.transpose(-1, -2)) / 2
    return M / (a.abs() ** 2).sum(-1)[..., None, None]


def principal_axis(M: torch.Tensor) -> torch.Tensor:
    """Unit eigenvector of the largest eigenvalue of each symmetric 3×3 matrix (batched eigh).   (…, 3, 3) → (…, 3)"""
    return torch.linalg.eigh(M).eigenvectors[..., -1]


def alignment(n2: torch.Tensor, n3: torch.Tensor) -> torch.Tensor:
    r"""|\hat n_2 \cdot \hat n_3| (headless axes): 1 = aligned, 0 = perpendicular.   (…, 3) → (…)"""
    return torch.einsum("...i,...i->...", n2, n3).abs()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/axis_of_evil/tensors.py
"""Build the concrete tensors: load the sky map, random rotations, isotropic random skies and a planted sky.

data/wmap_ilc9_nside16.csv is the WMAP 9-year Internal Linear Combination (ILC) temperature map (Bennett et al.
2013), degraded from HEALPix N_side = 512 to N_side = 16 (3072 equal-area pixels) by averaging the 1024 sub-pixels
of each pixel: Galactic longitude, latitude (degrees) and temperature (μK). Low resolution is plenty for ℓ = 2, 3.
"""

import math
from pathlib import Path

import torch

from . import equation

DATA = Path(__file__).parent / "data" / "wmap_ilc9_nside16.csv"
DTYPE = torch.float64
COLUMNS = {  # accepted header spellings (case-insensitive)
    "lon": ("glon_deg", "lon", "glon", "l", "longitude", "ra"),
    "lat": ("glat_deg", "lat", "glat", "b", "latitude", "dec"),
    "T": ("t_uk", "t", "temperature", "temp", "i", "i_stokes"),
    "w": ("weight", "w", "area_sr"),
}


def unit_vectors(lon_deg: torch.Tensor, lat_deg: torch.Tensor) -> torch.Tensor:
    """(lon, lat) in degrees → unit vectors (P, 3); x towards lon = 0, z towards lat = +90°."""
    lon, lat = torch.deg2rad(lon_deg), torch.deg2rad(lat_deg)
    return torch.stack([torch.cos(lat) * torch.cos(lon), torch.cos(lat) * torch.sin(lon), torch.sin(lat)], -1)


def lonlat(n: torch.Tensor) -> tuple[float, float]:
    """Unit vector → (longitude in [0, 360), latitude) in degrees."""
    lon = math.degrees(math.atan2(n[1].item(), n[0].item())) % 360
    return lon, math.degrees(math.asin(max(-1.0, min(1.0, n[2].item()))))


def load_sky_map(path: Path = DATA, device: torch.device | None = None):
    """Read a sky-map CSV: pixel directions (P, 3), temperatures (P,) and solid-angle weights (P,).

    Any comma-separated file with a header naming longitude, latitude (degrees) and temperature columns works
    (see COLUMNS; order free, extra columns ignored, # lines skipped), so an uploaded map can replace the bundled
    WMAP one. Without a weight column the pixels are taken as equal-area (HEALPix), each 4π/P steradians.
    """
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip() and ln[0] != "#"]
    header = [h.strip().lower() for h in lines[0].split(",")]
    index = {}
    for key, names in COLUMNS.items():
        found = [header.index(n) for n in names if n in header]
        if found:
            index[key] = found[0]
        elif key != "w":
            raise ValueError(f"no {key} column in {path}: expected one of {names}, got {header}")
    rows = [ln.split(",") for ln in lines[1:]]
    keys = [k for k in ("lon", "lat", "T", "w") if k in index]
    table = torch.tensor([[float(r[index[k]]) for k in keys] for r in rows], dtype=DTYPE, device=device)
    dirs = unit_vectors(table[:, 0], table[:, 1])
    w = table[:, 3] if "w" in index else torch.full_like(table[:, 0], 4 * math.pi / len(table))
    return dirs, table[:, 2], w


def random_rotation(generator: torch.Generator) -> torch.Tensor:
    """Uniformly random 3×3 rotation (QR of a Gaussian matrix with the sign fixed, det = +1)."""
    q, r = torch.linalg.qr(torch.randn(3, 3, dtype=DTYPE, generator=generator))
    q = q * torch.sign(torch.diagonal(r))
    return q if torch.det(q) > 0 else -q


def isotropic_coefficients(n_skies: int, ell: int, generator: torch.Generator) -> torch.Tensor:
    """Complex a_{ℓm} of n_skies isotropic Gaussian skies: (K, 2ℓ+1).

    Independent unit Gaussians on the real harmonic coefficients are exactly isotropic; the overall power C_ℓ
    does not matter because the axis only depends on the shape of a_m.
    """
    r = torch.randn(n_skies, 2 * ell + 1, dtype=DTYPE, generator=generator)
    return equation.real_to_complex(r, ell)


def planted_sky(dirs: torch.Tensor, axis: torch.Tensor, amplitudes=(30.0, 25.0)) -> torch.Tensor:
    r"""A sky whose quadrupole and octopole are both pure m = ±ℓ around the same axis: (P,).

    Re[(\hat n\cdot(e_1 + i e_2))^ℓ] with e_1, e_2 ⊥ axis is a sectoral harmonic: all its power is in m = ±ℓ.
    """
    helper = torch.tensor([1.0, 0.0, 0.0], dtype=DTYPE) if abs(axis[0]) < 0.9 else torch.tensor([0.0, 1.0, 0.0])
    e1 = torch.nn.functional.normalize(torch.linalg.cross(axis, helper.to(axis)), dim=0)
    e2 = torch.linalg.cross(axis, e1)
    zeta = dirs @ e1 + 1j * (dirs @ e2)  # (P,) complex
    return amplitudes[0] * (zeta**2).real + amplitudes[1] * (zeta**3).real


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/axis_of_evil/solve.py
r"""Fit the low multipoles of the map, find their axes, and test the alignment against isotropic random skies.

The a_{ℓm} come from a weighted least-squares fit of all harmonics up to ℓ_fit (one lstsq), which on a full sky
equals the usual quadrature a_{ℓm} = Σ_p w_p T_p Y^*_{ℓm}(\hat n_p) but also removes aliasing from the neighbouring ℓ.
The Monte Carlo p-value is the share of isotropic skies whose two axes are at least as aligned as the observed ones.
"""

import torch

from . import equation, tensors


def fit_alm(dirs: torch.Tensor, T: torch.Tensor, w: torch.Tensor, lmax: int) -> dict[int, torch.Tensor]:
    r"""Least squares min Σ_p w_p (T_p − Σ r_{ℓm} Y^{real}_{ℓm,p})² → {ℓ: complex a_{ℓm} (2ℓ+1,)} for ℓ ≤ lmax."""
    Y = equation.real_harmonics(dirs, lmax)  # (P, (lmax+1)²)
    sw = w.sqrt()
    r = torch.linalg.lstsq(Y * sw[:, None], (T * sw)[:, None]).solution[:, 0]  # ((lmax+1)²,)
    return {ell: equation.real_to_complex(r[ell**2 : (ell + 1) ** 2], ell) for ell in range(lmax + 1)}


def axis_of(a: torch.Tensor, ell: int):
    """Preferred axis (…, 3), its planarity t = max ⟨(n·L)²⟩/ℓ² ∈ [0, 1] and the gap to the 2nd eigenvalue."""
    M = equation.dispersion_tensor(a, ell)
    vals, vecs = torch.linalg.eigh(M)
    return vecs[..., -1], vals[..., -1] / ell**2, vals[..., -1] - vals[..., -2]


def alignment_of(alm: dict[int, torch.Tensor]) -> dict:
    """Axes of the quadrupole and octopole, the angle between them and |cos|."""
    n2, t2, g2 = axis_of(alm[2], 2)
    n3, t3, g3 = axis_of(alm[3], 3)
    cos = equation.alignment(n2, n3).item()
    return {
        "axis2": n2,
        "axis3": n3,
        "cos": cos,
        "angle": torch.rad2deg(torch.acos(torch.tensor(min(cos, 1.0)))).item(),
        "t2": t2.item(),
        "t3": t3.item(),
        "gap2": g2.item(),
        "gap3": g3.item(),
    }


def monte_carlo(n_skies: int, generator: torch.Generator) -> torch.Tensor:
    """|n̂_2 · n̂_3| for n_skies isotropic Gaussian skies at once: (K,)."""
    a2 = tensors.isotropic_coefficients(n_skies, 2, generator)  # (K, 5)
    a3 = tensors.isotropic_coefficients(n_skies, 3, generator)  # (K, 7)
    return equation.alignment(axis_of(a2, 2)[0], axis_of(a3, 3)[0])


def p_value(cos_obs: float, cos_sims: torch.Tensor) -> float:
    """Share of simulated skies at least as aligned as observed (for isotropy this tends to 1 − |cos|)."""
    return (cos_sims >= cos_obs).double().mean().item()


def power(alm: dict[int, torch.Tensor]) -> dict[int, float]:
    r"""Band power D_ℓ = ℓ(ℓ+1) C_ℓ / 2π in μK², with C_ℓ = Σ_m |a_{ℓm}|² / (2ℓ+1)."""
    return {ell: ell * (ell + 1) * (a.abs() ** 2).mean().item() / (2 * torch.pi) for ell, a in alm.items()}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/axis_of_evil/plot.py
"""Quadrupole and octopole maps with their axes, and the Monte Carlo test filling up frame by frame."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation, tensors


def multipole_map(a: torch.Tensor, ell: int, n_lat: int = 60, n_lon: int = 120):
    """T_ℓ(lon, lat) = Σ_m a_{ℓm} Y_{ℓm} on a lon/lat grid, lon from +180° to −180° (Galactic maps grow to the left)."""
    lon = torch.linspace(180, -180, n_lon, dtype=torch.float64)
    lat = torch.linspace(-90, 90, n_lat, dtype=torch.float64)
    n = tensors.unit_vectors(lon[None, :].expand(n_lat, -1), lat[:, None].expand(-1, n_lon))
    Y = equation.spherical_harmonics(n.reshape(-1, 3), ell)[:, ell * ell :]  # only this ℓ
    T = (Y @ a).real.reshape(n_lat, n_lon)
    return np.radians(-lon.numpy()), np.radians(lat.numpy()), T.numpy()


def _marks(ax, n, style, color, label, **kw):
    for sign in (1, -1):  # an axis has two ends
        lon, lat = tensors.lonlat(sign * n)
        lon = (lon + 180) % 360 - 180
        ax.plot(math.radians(-lon), math.radians(lat), style, color=color, label=label if sign == 1 else None, **kw)


def animate(alm, result, cos_sims, path: Path) -> Path:
    fig = plt.figure(figsize=(9.2, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.15, 1])
    for row, ell in enumerate((2, 3)):
        ax = fig.add_subplot(grid[row, 0], projection="mollweide")
        x, y, T = multipole_map(alm[ell], ell)
        lim = np.abs(T).max()
        ax.pcolormesh(x, y, T, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto", rasterized=True)
        _marks(ax, result["axis2"], "o", ACCENT[2], "quadrupole axis", ms=12, mfc="none", mew=2)
        _marks(ax, result["axis3"], "s", ACCENT[4], "octopole axis", ms=5, mec="white", mew=0.6)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.grid(color="#9ca3af", lw=0.4)
        ax.set_title(f"ℓ = {ell} (WMAP 9-yr ILC), ±{lim:.0f} μK", fontsize=8)
        if row == 1:
            ax.legend(fontsize=6.5, loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)

    ax_p = fig.add_subplot(grid[:, 1])
    sims = np.sort(1 - cos_sims.numpy())  # (K,) 1 − |cos| of the isotropic skies
    K = len(sims)
    x_obs = 1 - result["cos"]
    xs = np.geomspace(1e-5, 1, 60)
    ax_p.loglog(xs, xs, color="#6b7280", lw=1, ls="--", label="isotropy: P = 1 − |cos| (uniform)")
    mc = ax_p.loglog([], [], "o", ms=3.5, color=ACCENT[0], label="random Gaussian skies")[0]
    ax_p.axvline(x_obs, color=ACCENT[1], lw=1.5, label=f"WMAP: angle {result['angle']:.1f}°")
    ax_p.set_xlim(1e-5, 1)
    ax_p.set_ylim(1e-5, 1.5)
    ax_p.set_xlabel(r"$1 - |\hat n_2\cdot\hat n_3|$  (0 = perfectly aligned)")
    ax_p.set_ylabel("share of skies at least this aligned")
    ax_p.legend(fontsize=7, loc="upper left")
    title = ax_p.set_title("", fontsize=8.5)
    probes = np.geomspace(1e-5, 1, 26)
    sizes = [max(200, int(K * (i + 1) / 60)) for i in range(60)]

    def update(frame: int):
        k = min(sizes[frame], K)
        s = np.sort(sims[:k]) if k < K else sims
        share = np.searchsorted(s, probes, side="right") / k  # empirical P(1 − |cos| ≤ x)
        keep = share > 0
        mc.set_data(probes[keep], share[keep])
        p = np.mean(s <= x_obs)
        title.set_text(f"{k:,} isotropic skies: p = {p:.4f}  (one-sided, no look-elsewhere)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(60, 60), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_skies': 100000,
    'lmax_fit': 10,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.axis_of_evil")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.axis_of_evil.main")
problem_dir = Path("problems/cosmology/axis_of_evil")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
